# ECON2041 Week 12 tutorial: checking a regression's assumptions

### What you'll be able to do by the end

- Plot the residuals of a regression against its fitted values and look for a pattern
- Use the RESET test to check the functional form of a regression
- Use the Breusch-Pagan test to check whether the variance of the error term is the same for every observation
- Report robust standard errors, and say what they change and what they leave the same

### Different types of cells

- 🟢 Read and run: these cells have pre-written code, you can simply run them, read and interpret the output
- ✏️ You write: your turn to try to write code or sometimes add a short written answer
- 🤔 You think: stop and think on your own before running the next cell or reading on
- 💬 Discuss: talk with your neighbors before typing or running

## Setup

Our usual setup block, plus one line from the week 11 live lecture that imports the two tests that we'll run today: `linear_reset` for the RESET test and `het_breuschpagan` for the Breusch-Pagan test.

In [ ]:
# 🟢 Read and run: our standard ECON2041 setup block
import numpy as np               # numerical tools (nicknamed np)
import pandas as pd              # data tools (nicknamed pd)
import matplotlib.pyplot as plt  # plotting tools (nicknamed plt)
import seaborn as sns            # statistical charts (nicknamed sns)
from statsmodels.formula.api import ols  # ordinary least squares regression
from statsmodels.stats.diagnostic import het_breuschpagan, linear_reset  # the Breusch-Pagan and RESET tests

# Keep scalar output plain under NumPy 2 (0.5, not np.float64(0.5)).
if np.lib.NumpyVersion(np.__version__) >= "2.0.0":
    np.set_printoptions(legacy="1.25")

DATA = "https://emiliatjernstrom.com/econ2041/data"   # Unit datasets live at this web address

print("Setup done!")

## The pokies data

The dataset has one row for each of the 79 local government areas (LGAs) in Victoria, for 2024, as in the live lectures.
We'll use three of its variables:

- `exp_per_adult`: gambling losses on pokies per adult, in dollars (the dataset calls them net expenditure, the industry's phrase for what players lose)
- `egm_per_1000`: pokie density, the number of electronic gaming machines (EGMs, or pokies) per 1,000 adults
- `SEIFA_dis_score`: the Socio-Economic Indexes for Areas (SEIFA) disadvantage score, where a higher score means less disadvantaged

*Source:* Victorian Gambling and Casino Control Commission, gaming expenditure by LGA, 2024; Department of Environment, Land, Water and Planning, population, 2023; Australian Bureau of Statistics, SEIFA.

In [ ]:
# 🟢 Read and run: load pokies-victoria-2024.csv into a dataframe called pokies and look at the first few rows
pokies = pd.read_csv(f"{DATA}/pokies-victoria-2024.csv")

pokies.head()

## Question 1: are losses linear in pokie density?

In the week 6 live lecture, we regressed losses per adult on pokie density.
Every regression assumes a functional form: how each explanatory variable enters the regression.
Here, pokie density enters as itself, so the regression assumes that losses are linear in pokie density: one extra pokie per 1,000 adults is associated with the same difference in losses at every density.

As in the week 11 live lecture, we'll check the functional form with a residual plot, and then with the RESET test.

In [ ]:
# 🟢 Read and run: regress exp_per_adult on egm_per_1000, store the result as line_fit,
#                  and show the regression table rounded to 3 decimals
line_fit = ols("exp_per_adult ~ egm_per_1000", data=pokies).fit()

line_fit.summary2().tables[1].round(3)

In [ ]:
# 🟢 Read and run: plot the residuals of line_fit against its fitted values,
#                  with a smooth red curve through the residuals and a dashed line at 0
sns.regplot(x=line_fit.fittedvalues, y=line_fit.resid, lowess=True,
            scatter_kws={"edgecolor": "black"}, line_kws={"color": "red"})
plt.axhline(0, color="gray", linestyle="--")
plt.xlabel("Fitted losses per adult (AUD)")
plt.ylabel("Residual (AUD)")
plt.show()

The red curve shows the average residual of the LGAs with similar fitted values.
If the functional form is right, the residuals have no pattern, so the red curve stays close to 0.

🤔 Is the red curve close to 0 at every fitted value, or is there a pattern?

In [ ]:
# ✏️ You write: run the RESET test on line_fit, adding the square and the cube of the fitted values
# Hint: linear_reset takes the fitted model, line_fit, then power=3, which adds the square and the cube of the fitted values,
#       and use_f=True, which reports the F test of whether both of their coefficients are 0

🤔 Is the p-value below 0.1? What do we conclude about the functional form of `line_fit`?

## Question 2: what if we hold disadvantage fixed?

Losses could also depend on how disadvantaged an area is, and more disadvantaged areas could also have more pokies.
If so, leaving `SEIFA_dis_score` out of the regression biases the slope on pokie density, as in the week 7 Essential Concepts recordings on omitted variable bias.

In [ ]:
# ✏️ You write: regress exp_per_adult on egm_per_1000 and SEIFA_dis_score, store the result as seifa_fit,
# and show the regression table rounded to 3 decimals
# Hint: the same pattern as line_fit, with SEIFA_dis_score added after a +

In [ ]:
# ✏️ You write: run the RESET test on seifa_fit
# Hint: the same line as in question 1, with seifa_fit instead of line_fit

🤔 Does RESET still reject the null hypothesis at 0.1? Did adding `SEIFA_dis_score` fix the functional form?

## Question 3: add the square of pokie density

The residual plot in question 1 showed predicted losses that were too high in the LGAs with the most pokies.
We would see that pattern if one extra pokie per 1,000 adults is associated with a smaller difference in losses where density is already high.

Adding the square of pokie density to the regression allows for that, as in the week 11 Essential Concepts recordings:

$$\text{losses} = \beta_0 + \beta_1 \text{density} + \beta_2 \text{density}^2 + \beta_3 \text{SEIFA} + \varepsilon$$

With the square in the regression, the slope on pokie density is different at different densities.
If $\beta_2$ is negative, the slope is smaller where density is high.

In Python, `**` raises a number to a power, so `egm_per_1000**2` is the square of pokie density.
In an `ols` formula, we wrap it in `I()`, as in `I(egm_per_1000**2)`, so that `ols` computes the square before running the regression; without `I()`, `ols` would leave the square out.

In [ ]:
# ✏️ You write: regress exp_per_adult on egm_per_1000, its square, and SEIFA_dis_score, store the result as sq_fit,
# and show the regression table rounded to 3 decimals
# Hint: add I(egm_per_1000**2) to the formula of seifa_fit

In [ ]:
# ✏️ You write: plot the residuals of sq_fit against its fitted values, as in question 1
# Hint: copy the plot cell from question 1, and change line_fit to sq_fit in both places

In [ ]:
# ✏️ You write: run the RESET test on sq_fit
# Hint: the same line as for seifa_fit, with sq_fit instead

🤔 Can we reject the null hypothesis at 0.1 now? Does that prove that the quadratic is the right functional form?

## Question 4: is the variance of the error term the same for every LGA?

The OLS assumption of homoskedasticity says that the error term has the same variance for every observation, whatever the values of the explanatory variables.
Look back at the residual plot of `sq_fit`: are the dots spread equally widely around 0 at every fitted value?

The Breusch-Pagan test from the week 11 live lecture regresses the squared residuals on the explanatory variables, and tests the null hypothesis of homoskedasticity.
`het_breuschpagan` prints four numbers: the first is the LM statistic and the second is its p-value.
The last two are an F version of the same test, which we can skip.

In [ ]:
# ✏️ You write: run the Breusch-Pagan test on sq_fit
# Hint: het_breuschpagan takes two inputs, separated by a comma: the residuals, sq_fit.resid,
#       and the explanatory variables, sq_fit.model.exog

🤔 Is the p-value below 0.1? What do we conclude about the variance of the error term, and what does that mean for the usual standard errors in `sq_fit`?

In [ ]:
# ✏️ You write: fit the same model as sq_fit with robust standard errors, and store the result as robust_fit
# Hint: the same ols line as sq_fit, with cov_type="HC3" inside fit()

In [ ]:
# 🟢 Read and run: compare the coefficients, standard errors, and p-values of sq_fit and robust_fit, rounded to 3 decimals
pd.DataFrame({"Coef.": sq_fit.params,
              "Usual SE": sq_fit.bse, "Robust SE": robust_fit.bse,
              "Usual p": sq_fit.pvalues, "Robust p": robust_fit.pvalues}).round(3)

💬 With your neighbors: what is the same in the two versions, and what is different?
Do the robust standard errors change any conclusion at the unit's threshold of 0.1?